# Agentic RAG evaluation demo (for interview)

This notebook runs **RAG evaluation** on FinQA and TAT-QA and **displays results** for presenting. It does **not** build indexes or clone the repo.

**Prerequisites:** Clone the repo, install dependencies, download datasets, and **build retriever indexes** (run **build_retriever_index.ipynb** on Colab, then download and unzip the index bundles locally). Ensure `data/rag/FinQA/train/finqa_retriever_index/` and `data/rag/TAT-QA/tatqa_retriever_index/` exist.

1. Set **ANTHROPIC_API_KEY** from Colab Secrets.
2. Run RAG evaluation: **FinQA** (section 2).
3. Run RAG evaluation: **TAT-QA** (section 3).
4. Display results: FinQA (section 4) and TAT-QA (section 5).
6. Download per-sample JSON and predictions (section 6).
7. Lessons learned (section 7, RAG_LESSONS.md).


## What "agentic" RAG means here

**Strict definition (research-y):** an **autonomous** agent that plans, calls tools, and executes multi-step workflows end-to-end without human approval on each step.

**Loose, job-description definition (this notebook):** an **orchestrated, multi-step, tool-using RAG pipeline** where the model plans, retrieves, sometimes reranks, and executes numerical programs, but a human still reviews and signs off on results.

- **Normal RAG:**
  - Encode query → retrieve top-k chunks → generate a single-shot answer.
- **Agentic RAG (here):**
  - Plan query handling steps.
  - Retrieve with **corpus scoping** (e.g., FinQA vs TAT-QA corpora).
  - Optionally **rerank** retrieved passages.
  - Generate answers with **numerical program hints + execution**, then surface everything for human review.


## Prerequisites

Before running this notebook:

- **Repo** cloned and dependencies installed (or run sections 1–2 of **build_retriever_index.ipynb** once).
- **Datasets** FinQA and TAT-QA downloaded (section 2 of build notebook).
- **Retriever indexes** built and available locally: run **build_retriever_index.ipynb** on Colab (sections 1–5), download the zipped index bundles (section 6), then unzip into `data/rag/FinQA/train/` and `data/rag/TAT-QA/` so that `eval_runner.py` can load them.
- **ANTHROPIC_API_KEY** set (section 1 below) for Claude API calls.


## 1. Set ANTHROPIC_API_KEY from Colab Secrets (Optional)

The evaluation pipeline uses **Anthropic Claude** via the `ANTHROPIC_API_KEY` environment variable.

In Colab:

1. Go to **Settings → Secrets** (or use the key icon in the left sidebar).
2. Add a secret named `ANTHROPIC_API_KEY` with your API key value.
3. The cell below reads that secret using `google.colab.userdata` and sets `os.environ["ANTHROPIC_API_KEY"]`.


In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")


## 6a. Run RAG evaluation: FinQA (Optional)

This runs the **agentic RAG** pipeline on the **FinQA** dataset only (`--dataset FinQA`). Proof artifacts and metrics are written under `data/proof/rag/finqa/`.


In [ ]:
!python eval_runner.py --max_split 1 --max_category 1 --dataset FinQA --debug --export_predictions_txt


## 3. Run RAG evaluation: TAT-QA (Optional)

This runs the **agentic RAG** pipeline on the **TAT-QA** dataset only (`--dataset TATQA`). Proof artifacts and metrics are written under `data/proof/rag/tatqa/`.


In [ ]:
!python eval_runner.py --max_split 1 --max_category 1 --dataset TATQA --debug --export_predictions_txt


## 4. Display results: FinQA (optional)

Shows **aggregate scores** (weighted-average metrics) and where to find per-sample predictions so you can see how FinQA performed.


In [ ]:
import json
from pathlib import Path

try:
    from IPython.display import Markdown, display
    import pandas as pd
    _use_display = True
except ImportError:
    _use_display = False

proof_dir = Path("data/proof/rag")
dataset_name = "finqa"
dataset_dir = proof_dir / dataset_name

if not dataset_dir.exists():
    print(f"No results for {dataset_name}. Run section 2 first.")
else:
    if _use_display:
        display(Markdown("### FinQA — scores and outputs"))
    print("Outputs under:", dataset_dir.resolve())
    weighted = dataset_dir / f"{dataset_name}_avg.json"
    if weighted.exists():
        with open(weighted) as f:
            m = json.load(f)
        print("\n--- Aggregate scores (weighted average) ---")
        if _use_display:
            rows = [
                (k, f"{v:.4f}" if isinstance(v, (int, float)) and v is not None else str(v))
                for k, v in sorted(m.items())
            ]
            display(pd.DataFrame(rows, columns=["Metric", "Value"]))
        else:
            for k, v in sorted(m.items()):
                print(f"  {k}: {v}")
    else:
        print("\n(No aggregate metrics file yet; check per-sample files below.)")
    n_total = 0
    pred_paths = []
    for split_dir in sorted(dataset_dir.iterdir()):
        if split_dir.is_dir():
            for f in split_dir.glob("*_per_sample_*.json"):
                with open(f) as fp:
                    n = len(json.load(fp))
                n_total += n
                print(f"Samples evaluated: {n} (split: {split_dir.name})")
            for f in split_dir.glob("*_predictions.txt"):
                pred_paths.append(f)
                break
    if n_total:
        print(f"Total samples: {n_total}")
    for p in pred_paths:
        print(f"Predictions file: {p}")


## 5. Display results: TAT-QA (optional)

Shows **aggregate scores** and where to find per-sample predictions for TAT-QA.


In [ ]:
from pathlib import Path
import json

proof_dir = Path("data/proof/rag")
try:
    _use_display
except NameError:
    try:
        from IPython.display import Markdown, display
        import pandas as pd
        _use_display = True
    except ImportError:
        _use_display = False

dataset_name = "tatqa"
dataset_dir = proof_dir / dataset_name

if not dataset_dir.exists():
    print(f"No results for {dataset_name}. Run section 3 first.")
else:
    if _use_display:
        display(Markdown("### TAT-QA — scores and outputs"))
    print("Outputs under:", dataset_dir.resolve())
    weighted = dataset_dir / f"{dataset_name}_avg.json"
    if weighted.exists():
        with open(weighted) as f:
            m = json.load(f)
        print("\n--- Aggregate scores (weighted average) ---")
        if _use_display:
            rows = [
                (k, f"{v:.4f}" if isinstance(v, (int, float)) and v is not None else str(v))
                for k, v in sorted(m.items())
            ]
            display(pd.DataFrame(rows, columns=["Metric", "Value"]))
        else:
            for k, v in sorted(m.items()):
                print(f"  {k}: {v}")
    else:
        print("\n(No aggregate metrics file yet; check per-sample files below.)")
    n_total = 0
    pred_paths = []
    for split_dir in sorted(dataset_dir.iterdir()):
        if split_dir.is_dir():
            for f in split_dir.glob("*_per_sample_*.json"):
                with open(f) as fp:
                    n = len(json.load(fp))
                n_total += n
                print(f"Samples evaluated: {n} (split: {split_dir.name})")
            for f in split_dir.glob("*_predictions.txt"):
                pred_paths.append(f)
                break
    if n_total:
        print(f"Total samples: {n_total}")
    for p in pred_paths:
        print(f"Predictions file: {p}")


## 8b. Download per-sample JSON and predictions for learning

Download **only** the artifacts that help you learn and improve the RAG pipeline: **per-sample JSON** (full detail per question) and **predictions .txt** (human-readable). Run this in Colab after sections 6a and 6b so you can inspect failures and align with **data/proof/RAG_LESSONS.md**.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

proof = Path("data/proof/rag")
for name, label in [("finqa", "FinQA"), ("tatqa", "TAT-QA")]:
    d = proof / name
    if not d.exists():
        continue
    for split_dir in sorted(d.iterdir()):
        if not split_dir.is_dir():
            continue
        for f in split_dir.glob("*_per_sample_*.json"):
            out = f"{name}_per_sample_{split_dir.name}.json"
            shutil.copy(f, out)
            files.download(out)
            print(f"Downloaded: {out}")
        for f in split_dir.glob("*_predictions.txt"):
            out = f"{name}_predictions_{split_dir.name}.txt"
            shutil.copy(f, out)
            files.download(out)
            print(f"Downloaded: {out}")
        break
print("Use these files with data/proof/RAG_LESSONS.md to improve the pipeline.")

## 7. Lessons learned (RAG pipeline)

Full lessons (pipeline fixes, Colab run takeaways, research-backed improvements) are in **data/proof/RAG_LESSONS.md** in the repo. Summary:

- **Metrics:** Use strict numerical match for FinQA; primary metric is `numerical_exact_match` (and strict `exact_match` for numerics).
- **Pipeline:** First step is always RAG retrieval; pass `corpus_id` to scope retrieval; format chunk text clearly in the prompt; use more chunks when scoped to one doc.
- **Numerical QA:** Program synthesis + execution (FinQA format) is more reliable than free-form number-in-text; align output format (e.g. growth rate vs raw difference, % vs decimal) with gold.
- **Debugging:** Run with `--debug` and check `after_filter`, `num_chunks`, `has_table_like_content`; use per_sample JSON and predictions .txt (section 8b) to inspect failures.
- **Next steps:** Few-shot examples, "Numbers you may use" in prompt, table-preserving chunking, two-phase retrieval, number-aware re-ranking (see RAG_LESSONS.md).